In [3]:
import os
import torch # For building the networks 
import torchtuples as tt # Some useful functions
from torch import nn
import torch.nn.functional as F
from pycox.datasets import support
import numpy as np
import cox
from cox import CoxPH
from pycox.preprocessing.feature_transforms import OrderedCategoricalLong
import pandas as pd
import model
from model import Cox_cluster
import pandas as pd
import numpy as np
import sklearn_pandas
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn_pandas import DataFrameMapper
import seaborn as sns
from pycox.evaluation import EvalSurv
sns.set_style("white")
from pycox.preprocessing.feature_transforms import OrderedCategoricalLong
import cox
from cox import CoxPH
from pycox.preprocessing.feature_transforms import OrderedCategoricalLong
import model
from model import Cox_cluster

file_name_train=os.listdir(r"user path/var2_5/train")
file_name_test=os.listdir(r"user path/var2_5/test")
file_name_val=os.listdir(r"user path/var2_5/val")

file_name_re_train=os.listdir(r"user path/var2_5/re_train")
file_name_re_test=os.listdir(r"user path/var2_5/re_test")
file_name_re_val=os.listdir(r"user path/var2_5/re_val")

def function_work(x):
    y = x.rsplit('.', 2)[-2]
    return ('log' not in x, int(y) if y.isdigit() else float('inf'), x)

csvFiles = file_name_train
csvFiles_train=sorted(sorted(csvFiles, key=function_work, reverse=False),key=len, reverse=False)
csvFiles = file_name_test
csvFiles_test=sorted(sorted(csvFiles, key=function_work, reverse=False),key=len, reverse=False)
csvFiles = file_name_val
csvFiles_val=sorted(sorted(csvFiles, key=function_work, reverse=False),key=len, reverse=False)
csvFiles = file_name_re_train
csvFiles_re_train=sorted(sorted(csvFiles, key=function_work, reverse=False),key=len, reverse=False)
csvFiles = file_name_re_test
csvFiles_re_test=sorted(sorted(csvFiles, key=function_work, reverse=False),key=len, reverse=False)
csvFiles = file_name_re_val
csvFiles_re_val=sorted(sorted(csvFiles, key=function_work, reverse=False),key=len, reverse=False)
arr=[]
arr = [0 for i in range(100)] 
#csvFiles_test
arr_ibs=[]
arr_ibs = [0 for i in range(100)] 


In [5]:
import time

# Initialize variables to track total time
total_time = 0
for i in range(len(csvFiles_train)):
    start_time = time.time()  # Start tracking time
    file_path_train=r"user path/var2_5/train"+"/"+csvFiles_train[i]
    data1_train=pd.read_csv(file_path_train)
    file_path_test=r"user path/var2_5/test"+"/"+csvFiles_test[i]
    data1_test=pd.read_csv(file_path_test)
    file_path_val=r"user path/var2_5/val"+"/"+csvFiles_val[i]
    data1_val=pd.read_csv(file_path_val)
    file_path_re_train=r"user path/var2_5/re_train"+"/"+csvFiles_re_train[i]
    re_train=pd.read_csv(file_path_re_train)
    file_path_re_test=r"user path/var2_5/re_test"+"/"+csvFiles_re_test[i]
    re_test=pd.read_csv(file_path_re_test)
    file_path_re_val=r"user path/var2_5/re_val"+"/"+csvFiles_re_val[i]    
    re_val=pd.read_csv(file_path_re_val)
    df_test = data1_test
    df_train = data1_train
    df_val = data1_val
    cols_std = ['x1', 'x2', 'x3', 'x4','x5','x6','x7','x9','x10','x11','x12','x13','x14','x15']
    cols_ =  ['x8']
    standardize = [([col], StandardScaler()) for col in cols_std]
    leave = [(col, None) for col in cols_]
    x_mapper = DataFrameMapper(standardize+leave)
    x_train = x_mapper.fit_transform(df_train).astype('float32')
    x_test = x_mapper.transform(df_test).astype('float32')
    re_test = re_test
    re_train = re_train
    x_train = x_mapper.fit_transform(df_train).astype('float32')
    x_test = x_mapper.transform(df_test).astype('float32')
    x_val = x_mapper.transform(df_val).astype('float32')
    re_test = re_test
    re_train = re_train
    re_val=re_val
    cols_bin_re=list(re_train.columns)[1:]
    leave_re = [(col, None) for col in cols_bin_re]
    x_mapper_re= DataFrameMapper(leave_re)
    re_train = x_mapper_re.fit_transform(re_train).astype('float32')
    re_test = x_mapper_re.transform(re_test).astype('float32')
    re_val = x_mapper_re.transform(re_val).astype('float32')
    get_target = lambda df: (df['o'].values, df['delta'].values)
    y_train = get_target(df_train)
    y_val = get_target(df_val)
    durations_test, events_test = get_target(df_test)
    val = x_val,re_val, y_val
    optimizer = tt.optim.AdamWR(decoupled_weight_decay=0.001, cycle_eta_multiplier=0.5,
                            cycle_multiplier=2)
    In_Nodes= x_train.shape[1]
    num_nodes = [64, 64]
    Pathway_Nodes=num_nodes[0]
    Hidden_Nodes=num_nodes[1]
    Out_Nodes= 1
    batch_norm = True
    dropout = 0.2
    output_bias = False
    batch_size =128
    net= Cox_cluster(In_Nodes, Pathway_Nodes, Hidden_Nodes, Out_Nodes,)
    model = cox.CoxPH(net, optimizer)
    epochs = 100
    callbacks = [tt.callbacks.EarlyStopping()]
    verbose = True
    lrfind = model.lr_finder(x_train, re_train,y_train,batch_size, tolerance=100)
    print(lrfind.get_best_lr())
    model.optimizer.set_lr(lrfind.get_best_lr())
    log = model.fit(x_train, re_train,y_train, batch_size, epochs, callbacks, verbose, val_data=val,
                val_batch_size=batch_size)
    baseline = model.compute_baseline_hazards(batch_size=100000)
    surv = model.predict_surv_df(x_test,re_test,baseline_hazards_=baseline)
    ev = EvalSurv(surv, durations_test, events_test, censor_surv='km')
    arr[i]=ev.concordance_td()
    print(ev.concordance_td())
    time_grid = np.linspace(durations_test.min(), durations_test.max(), 100)
    arr_ibs[i]=ev.integrated_brier_score(time_grid)
    print(arr_ibs[i])
    end_time = time.time()
    elapsed_time = end_time - start_time  # Calculate time taken for this round
    total_time += elapsed_time  # Accumulate the total time
    print(f"Round {i+1} took {elapsed_time:.2f} seconds.")

In [ ]:
import statistics
def Average(lst):
    return sum(lst) / len(lst)
  

average_c_index = Average(arr)
print(average_c_index)
average_ibs = Average(arr_ibs)
print(average_ibs)
sd_c_index = statistics.stdev(arr)
print(sd_c_index)
sd_ibs = statistics.stdev(arr_ibs)
print(sd_ibs)

In [ ]:
average_time_per_round = total_time / 100
print(f"Average computation time per round: {average_time_per_round:.2f} seconds")